# 最高精度ベース 7 パターン提出

**現最高**: `submission_blend_3way_025_025_050.csv` → **Public 0.76955**（count1 : BPR128 : stacking = 0.25 : 0.25 : 0.5）

この 3 本ブレンドをベースに、重みの微調整と 4 本目追加で **7 パターン** の提出ファイルを生成する。学習は行わず既存 CSV のみでブレンド。

| # | 内容 | 提出ファイル |
|---|------|-------------|
| 1 | count1 軽め・BPR128 重め（stacking 50% 固定） | submission_blend_3way_020_030_050.csv |
| 2 | count1 重め・BPR128 軽め（stacking 50% 固定） | submission_blend_3way_030_020_050.csv |
| 3 | stacking 55% | submission_blend_3way_022_023_055.csv |
| 4 | stacking 45% | submission_blend_3way_027_028_045.csv |
| 5 | 微調整（0.23:0.27:0.50） | submission_blend_3way_023_027_050.csv |
| 6 | 4 本ブレンド（similar/improvement 10%） | submission_blend_4way_020_020_050_010.csv |
| 7 | 4 本ブレンド（4 本目 15%） | submission_blend_4way_020_020_045_015.csv |

## セットアップ（既存 CSV のパスのみ）

In [7]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from lib.improvement_candidates import get_setup
from lib.submission import verify_submission, blend_n_submissions, blend_two_submissions

ctx = get_setup(seed=42, output_dir="outputs", use_best_pipeline=True)
test_ids = ctx.test["ID"].values

path_count1 = ctx.submissions_dir / "submission_2hop_bpr64_count1.csv"
path_bpr128 = ctx.submissions_dir / "submission_2hop_bpr128_only.csv"
path_stacking = ctx.submissions_dir / "submission_improvement_03_stacking.csv"

for name, p in [("count1", path_count1), ("BPR128", path_bpr128), ("stacking", path_stacking)]:
    print(f"  [{name}] {'OK' if p.exists() else 'なし'}: {p.name}")
print(f"提出先: {ctx.submissions_dir}")

  [count1] OK: submission_2hop_bpr64_count1.csv
  [BPR128] OK: submission_2hop_bpr128_only.csv
  [stacking] OK: submission_improvement_03_stacking.csv
提出先: outputs/submissions


## 7 パターン生成（3 本ブレンド 1〜5、4 本ブレンド 6〜7）

In [8]:
paths_3 = [path_count1, path_bpr128, path_stacking]

# 3 本ブレンド: (count1, BPR128, stacking) の重み
patterns_3way = [
    ([0.20, 0.30, 0.50], "submission_blend_3way_020_030_050.csv"),   # 1: count1 軽め・BPR128 重め
    ([0.30, 0.20, 0.50], "submission_blend_3way_030_020_050.csv"),   # 2: count1 重め・BPR128 軽め
    ([0.22, 0.23, 0.55], "submission_blend_3way_022_023_055.csv"),   # 3: stacking 55%
    ([0.27, 0.28, 0.45], "submission_blend_3way_027_028_045.csv"),   # 4: stacking 45%
    ([0.23, 0.27, 0.50], "submission_blend_3way_023_027_050.csv"),   # 5: 微調整
]

for weights, out_name in patterns_3way:
    out_path = ctx.submissions_dir / out_name
    r = blend_n_submissions(paths_3, weights, out_path, test_ids=test_ids)
    status = "OK" if r.get("ok") else r.get("message", "")
    print(f"  {out_name}  ({status})")

# 4 本目候補: similar/improvement_05 がなければ 2 本ブレンド（count1+BPR128）を生成して使用
path_4th = None
for cand in ["submission_similar_movies_reviewed.csv", "submission_improvement_05_scale_pos_weight.csv"]:
    p = ctx.submissions_dir / cand
    if p.exists():
        path_4th = p
        print(f"  4本目に使用: {cand}")
        break

if path_4th is None:
    path_2way = ctx.submissions_dir / "submission_blend_bpr64_count1_bpr128.csv"
    if not path_2way.exists():
        blend_two_submissions(path_count1, path_bpr128, path_2way, weight_a=0.65, test_ids=test_ids)
        print(f"  4本目用に作成: {path_2way.name} (0.65:0.35)")
    path_4th = path_2way
    print(f"  4本目に使用: {path_4th.name}")

paths_4 = [path_count1, path_bpr128, path_stacking, path_4th]
patterns_4way = [
    ([0.20, 0.20, 0.50, 0.10], "submission_blend_4way_020_020_050_010.csv"),   # 6: 4本目 10%
    ([0.20, 0.20, 0.45, 0.15], "submission_blend_4way_020_020_045_015.csv"),   # 7: 4本目 15%
]
for weights, out_name in patterns_4way:
    out_path = ctx.submissions_dir / out_name
    r = blend_n_submissions(paths_4, weights, out_path, test_ids=test_ids)
    status = "OK" if r.get("ok") else r.get("message", "")
    print(f"  {out_name}  ({status})")

  submission_blend_3way_020_030_050.csv  (OK)
  submission_blend_3way_030_020_050.csv  (OK)
  submission_blend_3way_022_023_055.csv  (OK)
  submission_blend_3way_027_028_045.csv  (OK)
  submission_blend_3way_023_027_050.csv  (OK)
  4本目に使用: submission_blend_bpr64_count1_bpr128.csv
  submission_blend_4way_020_020_050_010.csv  (OK)
  submission_blend_4way_020_020_045_015.csv  (OK)


## 提出ファイル一覧（本ノートで生成したもの）

In [9]:
out_names = [
    "submission_blend_3way_020_030_050.csv",
    "submission_blend_3way_030_020_050.csv",
    "submission_blend_3way_022_023_055.csv",
    "submission_blend_3way_027_028_045.csv",
    "submission_blend_3way_023_027_050.csv",
    "submission_blend_4way_020_020_050_010.csv",
    "submission_blend_4way_020_020_045_015.csv",
]
for name in out_names:
    p = ctx.submissions_dir / name
    if p.exists():
        v = verify_submission(p, ctx.test)
        print(f"  {name}  ({v.get('message', '')})")
    else:
        print(f"  {name}  (未生成)")

  submission_blend_3way_020_030_050.csv  (OK)


  submission_blend_3way_030_020_050.csv  (OK)
  submission_blend_3way_022_023_055.csv  (OK)
  submission_blend_3way_027_028_045.csv  (OK)
  submission_blend_3way_023_027_050.csv  (OK)
  submission_blend_4way_020_020_050_010.csv  (OK)
  submission_blend_4way_020_020_045_015.csv  (OK)
